# Ring-slot fitting: coefficients first, circuit qualification second

This passive reciprocal two-port is a positive example for Circulax's new `fit_model` → coefficients → `component_from_coefficients` workflow. Accuracy alone is not circuit admission: compare rational passivity and the stability of the **converted Y model**, not just S poles.

On the 160/41 training/holdout split below, scikit-rf's three-pole and automatic seven-pole fits fail rational passivity and have unstable Y poles. Its **four-pole fit passes**. The original Circulax seven-pole fit also fails; rational coefficient enforcement fixes those checks with a small accuracy penalty. This is a qualification comparison, not a claim that scikit-rf is unsuitable.

Data: scikit-rf's built-in [ring-slot example](https://scikit-rf.readthedocs.io/en/latest/examples/vectorfitting/vectorfitting_ex1_ringslot.html).

In [ ]:
import time
import warnings
import numpy as np
import matplotlib.pyplot as plt
import skrf
from skrf.vectorFitting import VectorFitting
from circulax.fitting import (
    ModelFitOptions, ModelCoefficients, fit_model, component_from_coefficients,
    scattering_state_space_to_admittance, surface_from_fit, validate_surface_fit,
)
from circulax.fitting.types import VFModel, vfmodel_to_ss

network = skrf.data.ring_slot
freqs, S = network.f, network.s
z0 = float(network.z0[0, 0].real)
train = np.arange(len(freqs)) % 5 != 0
holdout = ~train
print(f"{train.sum()} training / {holdout.sum()} held-out samples; {freqs[0]/1e9:g}–{freqs[-1]/1e9:g} GHz")


## Fit-only comparison

Use identical measured training data. The NumPy path selects order automatically; no pole count is prescribed. Timers below exclude conversion, passivity testing, validation, and plotting. Single-run times are illustrative, not warmed benchmark medians.

In [ ]:
baselines = {}
for label, order in [("scikit-rf fixed 3", 3), ("scikit-rf fixed 4", 4), ("scikit-rf auto", None)]:
    fitter = VectorFitting(network[train])
    started = time.perf_counter()
    if order is None:
        fitter.auto_fit()
    else:
        fitter.vector_fit(n_poles_real=order, n_poles_cmplx=0)
    elapsed = time.perf_counter() - started
    poles, residues = [], []
    for index, pole in enumerate(fitter.poles):
        residue = fitter.residues[:, index].reshape(2, 2)
        poles.append(pole)
        residues.append(residue)
        if pole.imag != 0:
            poles.append(pole.conjugate())
            residues.append(residue.conjugate())
    coefficients = ModelCoefficients(
        np.asarray(poles), np.stack(residues, axis=-1),
        fitter.constant_coeff.reshape(2, 2), z0,
    )
    baselines[label] = coefficients
    print(label, "poles:", len(poles), "fit ms:", elapsed * 1000)

started = time.perf_counter()
raw = fit_model(S[train], freqs[train], z0=z0, options=ModelFitOptions(
    normalized_rmse=8e-7, max_absolute_error=1e-5,
))
print("Circulax automatic:", len(raw.poles), "poles; fit ms:", (time.perf_counter() - started) * 1000)


## Enforce the rational coefficients

Keep the automatically selected seven poles fixed. With supplied poles and `iterations=0`, the public API refits residues and D, then applies optional rational enforcement. Allow a final training NRMSE of 3 ppm: the original 0.8 ppm target is too tight after correction.

A broad 4000-point grid is used here. **Grid convergence is not a global passivity certificate**; the next section separately runs scikit-rf's rational passivity test. No extrapolated measurements or held-out samples enter fitting/enforcement.

In [ ]:
started = time.perf_counter()
corrected = fit_model(
    S[train], freqs[train], z0=z0, initial_poles=raw.poles,
    options=ModelFitOptions(
        iterations=0, normalized_rmse=3e-6, max_absolute_error=1e-5,
        enforce_passivity=True,
        enforcement_freqs=tuple(np.r_[freqs[train], np.geomspace(1e5, 1e15, 4000)]),
    ),
)
print("Fixed-pole refit + enforcement ms:", (time.perf_counter() - started) * 1000)
print(corrected.metadata["enforcement"])
np.testing.assert_array_equal(corrected.poles, raw.poles)


## Test rational passivity and the actual Y realization

The same checks apply to every backend. A stable S pole set does not imply a stable Y realization: zeros of `det(I+S)` can become Y poles. The scikit-rf rational test also examines behavior beyond the measurement band. Report both properties, plus untouched holdout error.

In [ ]:
def rational_test(coefficients):
    oracle = VectorFitting(network[train])
    indices = np.flatnonzero(coefficients.poles.imag >= 0)
    oracle.poles = coefficients.poles[indices]
    oracle.residues = coefficients.residues[:, :, indices].reshape(4, -1)
    oracle.constant_coeff = coefficients.D.real.ravel()
    oracle.proportional_coeff = np.zeros(4)
    return oracle.passivity_test()

def admittance(coefficients):
    model = VFModel(coefficients.poles, coefficients.residues, coefficients.D.real, np.zeros((2, 2)))
    ss, _ = scattering_state_space_to_admittance(vfmodel_to_ss(model, 2), z0=z0)
    return ss

models = {**baselines, "Circulax raw 7": raw, "Circulax enforced 7": corrected}
qualification = {}
for label, coefficients in models.items():
    bands = rational_test(coefficients)
    ss = admittance(coefficients)
    error = np.linalg.norm(coefficients.evaluate(freqs[holdout]) - S[holdout]) / np.linalg.norm(S[holdout])
    max_y_pole = float(np.asarray(ss.A).real.max())
    qualification[label] = (len(bands) == 0, max_y_pole < 0)
    print(f"{label:24s} order={len(coefficients.poles)} holdout={error:.3e} "
          f"passive={not len(bands)} max Re(Y pole)={max_y_pole:.3e} rad/s")
    if len(bands):
        print("  passivity violation intervals (Hz):", bands)
assert qualification["scikit-rf fixed 4"] == (True, True)
assert qualification["Circulax enforced 7"] == (True, True)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=True)
predicted = corrected.evaluate(freqs)
for row in range(2):
    for col in range(2):
        ax = axes[row, col]
        ax.plot(freqs / 1e9, 20 * np.log10(np.maximum(abs(S[:, row, col]), 1e-15)), label="measured")
        ax.plot(freqs / 1e9, 20 * np.log10(np.maximum(abs(predicted[:, row, col]), 1e-15)), "--", label="enforced rational")
        ax.set_title(f"S{row + 1}{col + 1}")
        ax.set_xlabel("GHz")
        ax.set_ylabel("dB")
        ax.grid(True)
axes[0, 0].legend()
plt.tight_layout()


## Circulax circuit validation and component creation

Validate error, reciprocity, sampled admittance passivity, asymptotic terms, and Y stability before creating a component. This does not prove measured transient accuracy or validate unknown low-frequency behavior.

In [ ]:
ss = admittance(corrected)
features = np.ones((1, 1))
surface = surface_from_fit(ss, np.zeros(2), 2 * np.pi * freqs[-1], z0=z0)
report = validate_surface_fit(
    surface, S[train][None], features, freqs[train],
    validation_S=S[holdout][None], validation_features=features, validation_freqs=freqs[holdout],
    passivity_features=features, passivity_freqs=np.linspace(0, freqs[-1], 801),
    simulation_frequency_range=(0, freqs[-1]),
)
print(report.summary())
report.raise_for_simulation()
RingSlot = component_from_coefficients(corrected, name="RingSlot")
ring_slot_component = RingSlot()
print("Circulax component ports:", RingSlot.ports)


## Save coefficients and load in another process

The file stores S poles, residues, D, reference impedance, fit options, and diagnostics; component construction performs no fitting. The temporary archive below demonstrates the boundary without leaving an artifact in the repository.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as directory:
    archive = Path(directory) / "ring_slot.npz"
    corrected.save(archive)
    ReloadedRingSlot = component_from_coefficients(archive, name="ReloadedRingSlot")
    restored = ModelCoefficients.load(archive)
    np.testing.assert_allclose(restored.evaluate(freqs), corrected.evaluate(freqs))
print("Coefficient save/load and component generation: PASS")


## Suitability for Circulax

The enforced seven-pole model passes the tested rational passivity and Circulax circuit gates while retaining approximately 2.47 ppm held-out normalized RMS error, versus 0.72 ppm before enforcement. It is a qualified candidate for the tested use case, not a guarantee of accurate transients outside the measured 75–110 GHz band.

The unenforced scikit-rf three-pole and automatic seven-pole baselines are **not suitable as-is** for a passive Circulax admittance component: their Y realizations are unstable and their rational passivity tests fail. However, scikit-rf's four-pole fit passes these two physical checks. Scikit-rf also provides its own passivity enforcement; we have not compared enforcement performance here. The original Circulax seven-pole model fails too, so AAA alone is not the safety advantage.

Enforcement adds substantial cost and worsens the original tight accuracy target. The corrected Y feedthrough has an eigenvalue near 4×10⁴ S; passivity does not make this out-of-band behavior physically trustworthy. Component creation checks stability, not global passivity, so keep the independent tests.

See [API options](../../docs/fitting_api.md) and the [engineering explanation](../../docs/rational_model_enforcement.md). The active and resonant notebooks remain advanced stress cases, not automatic success claims.